In [ ]:
import pandas as pd

In [ ]:
# Load CSV file
df = pd.read_csv("../data/raw/saas_businesses_data.csv")

In [ ]:
print("Preview of the first five rows:")
display(df.head())

In [ ]:
data_types = df.dtypes
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df) * 100).round(2)

In [ ]:
# Combine into one summary DataFrame
summary = pd.DataFrame({
 "Data Type": data_types,
 "Missing Values": missing_values,
 "Missing %": missing_percent
})

In [ ]:
# Display summary
display(summary.sort_values(by="Missing %", ascending=False))

Step 3:

Goal: Data Cleaning + Feature Engineering

Cleaning missing values (fill/drop strategy)?

Or Engineering new features like:

profitMargin = profit / revenue

companyAge = 2023 - dateFounded

growthCategory = high/medium/low based on growth %

In [ ]:
# 1. Fill 'totalGrowthAnnual' using assignment, not inplace
df['totalGrowthAnnual'] = df['totalGrowthAnnual'].fillna(df['totalGrowthAnnual'].median())

In [ ]:
# 2. Fill 'location' with "Unknown"
df['location'] = df['location'].fillna("Unknown")

In [ ]:
# 3. Drop rows with missing 'listingHeadline'
df = df.dropna(subset=['listingHeadline'])

In [ ]:

# 4. Fill 'customers' and 'techStack' with placeholder
df['customers'] = df['customers'].fillna("Not Provided")
df['techStack'] = df['techStack'].fillna("Not Provided")

# Check again to confirm missing values are handled
print(" Remaining Missing Values:")
display(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Step 4: Feature engineering

import numpy as np
from datetime import datetime

# 1. Create 'profitMargin'
df['profitMargin'] = (df['totalProfitAnnual'] / df['totalRevenueAnnual']).round(2)

# 2. Convert 'dateFounded' to datetime and create 'companyAge'
df['dateFounded'] = pd.to_datetime(df['dateFounded'], errors='coerce')
df['companyAge'] = datetime.now().year - df['dateFounded'].dt.year

# 3. Create 'growthCategory' based on totalGrowthAnnual
def categorize_growth(x):
    if x >= 75:
        return 'High Growth'
    elif x >= 25:
        return 'Medium Growth'
    else:
        return 'Low Growth'

df['growthCategory'] = df['totalGrowthAnnual'].apply(categorize_growth)

# 4. Estimate valuation from revenue multiple
df['estimatedValuation'] = df['totalRevenueAnnual'] * df['revenueMultiple']

# Preview new columns
df[['profitMargin', 'companyAge', 'growthCategory', 'estimatedValuation']].head()


**Profit Margin and Growth**


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style


# 1. Average profit margin by growth category
plt.figure(figsize=(8,5))
sns.barplot(x='growthCategory', y='profitMargin', data=df)
plt.title(' Average Profit Margin by Growth Category')
plt.xlabel('Growth Category')
plt.ylabel('Profit Margin')
plt.tight_layout()
plt.show()


**Age and Country Insights**


In [ ]:
# 2. Company Age Distribution
plt.figure(figsize=(8,5))
sns.histplot(df['companyAge'], bins=15, kde=True)
plt.title('Company Age Distribution')
plt.xlabel('Company Age (Years)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Top 10 Countries by Average Asking Price
top_countries = df.groupby('location')['askingPrice'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
top_countries.plot(kind='barh')
plt.title(' Top 10 Countries by Average Asking Price')
plt.xlabel('Average Asking Price (USD)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


**Business Insights EDA**


**Are higher revenue multiples justified by higher growth?**


In [ ]:
# Step 6.1: Explore relationship between Revenue Multiple and Growth %

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='totalGrowthAnnual', y='revenueMultiple', hue='growthCategory', alpha=0.8)
sns.regplot(data=df, x='totalGrowthAnnual', y='revenueMultiple', scatter=False, color='black', line_kws={"linewidth":1.5})
plt.title(" Do High Growth Companies Have Higher Revenue Multiples?")
plt.xlabel("Total Growth Annual (%)")
plt.ylabel("Revenue Multiple")
plt.tight_layout()
plt.show()

# Check the correlation coefficient
correlation = df['totalGrowthAnnual'].corr(df['revenueMultiple'])
print(f" Correlation between Revenue Multiple and Growth: {correlation:.2f}")


The regression line is slightly flat or declining, meaning:

There is no strong positive relationship between annual growth rate and revenue multiple in this dataset.

High growth companies (even those with 5,000%+ growth) are not consistently getting higher revenue multiples.

Revenue multiple is likely influenced by other factors like:

Profitability

Market size

Monetization model

Founders' reputation, etc.

**Do More Profitable Companies Have Higher Asking Prices?**


In [ ]:
# Insight 2: Does Profit Margin Influence Asking Price?

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='profitMargin', y='askingPrice', hue='growthCategory', alpha=0.8)
sns.regplot(data=df, x='profitMargin', y='askingPrice', scatter=False, color='black', line_kws={"linewidth":1.5})
plt.title(" Does Profitability Drive Higher Asking Prices?")
plt.xlabel("Profit Margin")
plt.ylabel("Asking Price (USD)")
plt.tight_layout()
plt.show()

# Print correlation
correlation = df['profitMargin'].corr(df['askingPrice'])
print(f" Correlation between Profit Margin and Asking Price: {correlation:.2f}")



The regression line is almost flat, possibly slightly declining.

This indicates that:

Higher profit margins do NOT strongly correlate with higher asking prices.

**Are Newer SaaS Startups Growing Faster?**


In [ ]:
# Insight 3: Company Age vs Total Growth %

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='companyAge', y='totalGrowthAnnual', hue='growthCategory', alpha=0.8)
sns.regplot(data=df, x='companyAge', y='totalGrowthAnnual', scatter=False, color='black', line_kws={"linewidth":1.5})
plt.title("Do Younger SaaS Startups Grow Faster?")
plt.xlabel("Company Age (Years)")
plt.ylabel("Total Growth Annual (%)")
plt.tight_layout()
plt.show()

# Calculate correlation
correlation = df['companyAge'].corr(df['totalGrowthAnnual'])
print(f"Correlation between Company Age and Growth: {correlation:.2f}")


The regression line clearly slopes downward

This suggests a negative correlation: as company age increases, growth rate tends to decrease

A few younger companies show explosive growth (some >60,000%), likely early-stage breakout products

**Which Locations Offer the Strongest Profit-to-Price Ratio?**


In [ ]:
# Step 6.4: Which Locations Offer the Strongest Profit-to-Price Ratio?

# Avoid division by zero or NaNs
df = df[df['askingPrice'] > 0]

# Create acquisition return proxy metrics
df['profitToAskingPrice'] = df['totalProfitAnnual'] / df['askingPrice']
df['paybackYears'] = df['askingPrice'] / df['totalProfitAnnual']
df.loc[df['totalProfitAnnual'] <= 0, 'paybackYears'] = pd.NA

# Group by location and rank only locations with at least two listings
roi_by_country = (df[df['location'] != 'Unknown']
 .groupby('location')
 .agg(listings=('location', 'size'), median_profit_to_price=('profitToAskingPrice', 'median'))
 .query('listings >= 2')
 .sort_values('median_profit_to_price', ascending=False)
 .head(10))

# Plot
plt.figure(figsize=(10,6))
roi_by_country['median_profit_to_price'].plot(kind='barh')
plt.title(" Top Locations by Median Profit-to-Price Ratio (2+ Listings)")
plt.xlabel("Median Annual Profit / Asking Price")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


Location rankings should be treated as directional, not definitive. Countries with only one listing can appear attractive because of sample-size noise. For acquisition analysis, profit-to-price ratio and payback period are more useful than profit margin divided by asking price.

In [ ]:
# Save cleaned dataset for local use
output_path = "../data/processed/saas_cleaned_data.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to {output_path}")
